# VEP DNA Pipeline

In [23]:
%load_ext autoreload
%autoreload 2

import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True
    
import pandas as pd
import polars as pl
import seqpro as sp
import numpy as np
import pooch
from tqdm.auto import tqdm
from pathlib import Path
from tempfile import TemporaryDirectory

import genvarloader as gvl

import src.genvarloader as GVL

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create GVL database

## Download population data

In [2]:
# GRCh38 chromosome 22 sequence
reference = pooch.retrieve(
    url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
    known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
    progressbar=True,
)
if not Path(f"{reference[:-3]}.bgz").exists():
    !gzip -dc {reference} | bgzip > {reference[:-3]}.bgz
reference = reference[:-3] + ".bgz"

# PLINK 2 files
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.psam",
    known_hash="md5:eefa7aad5acffe62bf41df0a4600129c",
    progressbar=True,
    fname="1kGP.chr22.psam",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pvar",
    known_hash="md5:5f922af91c1a2f6822e2f1bb4469d12b",
    progressbar=True,
    fname="1kGP.chr22.pvar",
)

# BED
bed_path = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/chr22_egenes.bed",
    known_hash="md5:ccb55548e4ddd416d50dbe6638459421",
    progressbar=True,
)


## Create BED file

This BED file is based on the coordinates of the UTR variants we're planning on injecting, surrounded by 500kb windows.

In [3]:
bed = gvl.read_bedlike("data/UTR/snps_500kb_windows.bed")[:20]
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""22""",16835042,17335042,"""340557""",0.0,"""+"""
"""22""",16835050,17335050,"""340558""",0.0,"""+"""
"""22""",16835053,17335053,"""340559""",0.0,"""+"""
"""22""",16835059,17335059,"""894942""",0.0,"""+"""
"""22""",16835065,17335065,"""340560""",0.0,"""+"""


## Create GVL database

In [4]:
tmp_dir = TemporaryDirectory(suffix=".gvl")
ds_path = tmp_dir.name
gvl.write(
    path=ds_path,
    bed=gvl.with_length(bed, 2**17),  # change region length to 131,072 bp
    variants=variants,
    overwrite=True,
)

2025-05-16 23:52:59.702 | INFO     | genvarloader._dataset._write:write:76 - Writing dataset to /tmp/tmppnjporrr.gvl
2025-05-16 23:52:59.704 | INFO     | genvarloader._dataset._write:write:83 - Found existing GVL store, overwriting.
2025-05-16 23:52:59.730 | INFO     | genoray._pgen:_read_index:1054 - Loading genoray index.
2025-05-16 23:53:00.808 | INFO     | genvarloader._dataset._write:write:151 - Using 451 samples.
2025-05-16 23:53:00.809 | INFO     | genvarloader._dataset._write:write:157 - Writing genotypes.
Processing genotypes for 20 regions on contig 22: 100%|██████████| 20/20 [00:01<00:00, 18.45 region/s]
2025-05-16 23:53:02.126 | INFO     | genvarloader._dataset._write:write:181 - Finished writing.
/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datet

## Import GVL database

In [5]:
ds = (
    gvl.Dataset.open(ds_path, reference=reference)
    .with_seqs("haplotypes")
    .with_len(2**17)
)

2025-05-16 23:53:04.378 | INFO     | genvarloader._dataset._impl:open:215 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-16 23:53:04.643 | INFO     | genvarloader._dataset._reconstruct:from_path:298 - Loading variant data.
2025-05-16 23:53:04.678 | INFO     | genvarloader._dataset._impl:open:302 - Opened dataset:
GVL store at /tmp/tmppnjporrr.gvl
Is subset: False
# of regions: 20
# of samples: 451
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None



## Run VEP

Initialize xarray Dataset.

In [46]:
import xarray as xr

# Set environment variable to suppress datetime warnings
os.environ['PYTHONWARNINGS'] = 'ignore::DeprecationWarning:jupyter_client.session'

results_dir = 'vep_results'
variant_set = "ClinVar_UTR"
force = True
models = ["flashzoi", "evo2-7b", "spliceai_mm"]

# Create path for results file
zarr_path = os.path.join(results_dir, f"{variant_set}.zarr")
os.makedirs(results_dir, exist_ok=True)

sites = gvl.sites_vcf_to_table('./data/UTR/filtered_chr22_snps.vcf')
site_ds = gvl.DatasetWithSites(ds, sites) 
# Add the site_name column
site_ds.rows = site_ds.rows.with_columns(
    (pl.lit("chr") + pl.col("chrom") + pl.lit(":") + 
    pl.col("chromStart").cast(pl.Utf8) + pl.lit("-") + 
    pl.col("chromEnd").cast(pl.Utf8)).alias("site_name") +
    pl.lit("_") + pl.col("REF") +
    pl.lit("_") + pl.col("ALT")
)

# Define the Dataset variables
all_samples = site_ds.dataset.samples
all_sites = site_ds.rows[1:]["site_name"].to_list() # Skip the first row (WT)
all_ploid = [0,1]
all_slots = ["VEP"]

# Check if output file already exists and we're not forcing overwrite
if os.path.exists(zarr_path) and not force:
    print(f"Loading existing results from {zarr_path}")
    ds_results = xr.open_dataset(zarr_path)
else:
    print(f"Will save results to {zarr_path}")

    # xarray dataset structure
    # --> Model_name
    #   --> Site_name
    #       --> Sample_name
    #           --> Ploidy  
    #               --> VEP
    #               --> [other slots]

    # Initialize the scores array
    data_vars = {}
    for model_name in models:
        # Create a numpy array with dimensions [sites, samples, ploidy, slots]
        data_array = np.empty((len(all_sites), 
                                len(all_samples), 
                                len(all_ploid), 
                                len(all_slots)), 
                                dtype=object)
        # Fill with None values
        data_array.fill(None) 
        # Store the array with its dimension information
        dims = ['site', 'sample', 'ploid', 'slot'] 
        data_vars[model_name] = xr.DataArray(
            data_array, 
            dims=dims,
            coords=dict(zip(dims, [all_sites, all_samples, all_ploid, all_slots]))
        )
        
    # Create the xarray dataset with proper coordinates
    ds_results = xr.Dataset(
        data_vars=data_vars
    )
    # Save to zarr file
    os.makedirs(os.path.dirname(zarr_path), exist_ok=True)
    ds_results.to_zarr(zarr_path, mode="w")
    print(f"Results saved to {zarr_path}")


[E::idx_find_and_load] Could not retrieve index file for './data/UTR/filtered_chr22_snps.vcf'
[E::idx_find_and_load] Could not retrieve index file for './data/UTR/filtered_chr22_snps.vcf'
[E::idx_find_and_load] Could not retrieve index file for 'data/UTR/filtered_chr22_snps.vcf'
Reading records: 0 record [00:00, ? record/s][W::vcf_parse] Contig '22' is not defined in the header. (Quick workaround: index the file with tabix.)
Reading records: 5529 record [00:00, 293559.34 record/s]
/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Will save results to vep_results/ClinVar_UTR.zarr


/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Results saved to vep_results/ClinVar_UTR.zarr


In [47]:
ds_results

<xarray.Dataset> Size: 74MB
Dimensions:      (site: 3419, sample: 451, ploid: 2, slot: 1)
Coordinates:
  * site         (site) <U27 369kB 'chr22:17019506-17150578_G_A' ... 'chr22:1...
  * sample       (sample) <U7 13kB 'HG00096' 'HG00097' ... 'NA20826' 'NA20828'
  * ploid        (ploid) int64 16B 0 1
  * slot         (slot) <U3 12B 'VEP'
Data variables:
    flashzoi     (site, sample, ploid, slot) object 25MB None None ... None None
    evo2-7b      (site, sample, ploid, slot) object 25MB None None ... None None
    spliceai_mm  (site, sample, ploid, slot) object 25MB None None ... None None

Populate the xarray dataset

In [48]:
import src.vep_pipeline as vp

force = False
# Iterate over each site, which are centered on the SNPs
## Useing .rows only accesses the variants that overlap at least one region
models = ["spliceai_mm"]

# Iterate over models
for model_name in models:
    model_name = model_name.lower()
    
    # Load the model
    model = vp.load_model(model_name)
    tokenizer = vp.load_tokenizer(model_name)

    for site in tqdm(site_ds.rows.iter_rows(named=True),
                    total=site_ds.n_rows,
                    desc="Iterating over sites"):
        site_idx = site['site_idx']
        
        # Skip the first row in sites (WT)
        if site_idx == 0:
            continue

        # Iterate over each sample
        for sample_idx, sample in tqdm(enumerate(all_samples),
                                    total=len(all_samples),
                                    desc="Iterating over samples",
                                    leave=False): 
            
            region_idx = site_ds.rows[sample_idx, "region_idx"]

            # Get the WT haplotype
            seq_wt = GVL.get_wt_haps(site_ds, sample_idx)
            
            # Generate the mutated sequence
            ann_haps, flags = site_ds[site_idx, sample_idx]
            seq_mut = ann_haps.haps

            # Iterate over ploidy
            for ploid_idx in all_ploid:
                seq_wt = site_ds.dataset[region_idx, sample_idx].haps[ploid_idx]
                seq_mut = ann_haps.haps[ploid_idx]
            
                
                # Skip if the value is already set
                current_value = ds_results[model_name][site_idx , sample_idx, ploid_idx, 0]
                if current_value is not None and not force:
                    continue 
                
                # Run the model
                vep = vp.run_vep(model_name=model_name,
                                 model=model,
                                 tokenizer=tokenizer,
                                 seq_wt=seq_wt, 
                                 seq_mut=seq_mut)
                
                # Store the VEP score
                ds_results[model_name][site_idx , sample_idx, ploid_idx, 0] = vep
        
        # --- Limit the number of samples and sites to run for testing --- #
            if sample_idx>50:
                break
        if site_idx>10:
            break


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RnaTokenizer'. 
The class this function is called from is 'DnaTokenizer'.
Iterating over sites:   0%|          | 11/3420 [00:05<26:04,  2.18it/s]


Update the Zarr file

In [156]:
# Available modes for to_zarr:
# - "w": Create new store, overwrite if exists (default)
# - "a": Append to existing store
# - "r": Read-only access
# - "w-": Create new store, fail if exists
ds_results.to_zarr(zarr_path, mode="a")


/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Get the unique haplotypes

In [ ]:
import awkward as ak
import numba as nb
import numpy as np
from attrs import define
from seqpro._ragged import Ragged
from genvarloader._dataset._reconstruct import Haps, V_IDX_TYPE
from genvarloader._dataset._impl import Dataset


@define
class UniqueInfo:
    """Class to store unique genotypes."""

    # sample, ploidy => ds[region_idx, sample_idx][ploid_idx] (ploidy length) => unique haplotype sequence
    first_idxs: Ragged[np.void]
    """First haplotype indices. Shape: (n_regions, ~n_unique)"""
    inverse_idxs: ak.Array  # total elements = regions * samples * ploidy
    """Inverse indices. Shape: (n_regions, ~n_unique, ~n_haps)"""
    counts: Ragged[np.uint32]
    """Counts. Shape: (n_regions, ~n_unique)"""

    @property
    def n_regions(self) -> int:
        """Number of regions."""
        return self.first_idxs.shape[0]

    @property
    def n_unique(self):
        """Number of unique genotypes."""
        return self.first_idxs.lengths


def unique(dataset: Dataset) -> UniqueInfo:
    if not isinstance(dataset._seqs, Haps):
        raise ValueError(
            "Dataset must have genotypes/haplotypes to compute unique haplotype information."
        )

    genos = dataset._seqs.genotypes
    regions = dataset._full_regions
    out_len = dataset.output_length
    first_idxs, inverse_idxs, counts = _unique_genotypes(
        genos.data, genos.offsets, regions, out_len
    )


# @nb.njit(nogil=True, cache=True)
# def _unique_genotypes(v_idxs, v_offsets, regions, out_len): ...